# 03 — Gold Layer
Aggregates Silver data into 3 KPI tables for business reporting.
Manhattan has 88% of trips causing data skew — handled using 4-bucket salting.
Z-ORDER applied on key query columns to minimize files scanned.

In [0]:
client_id     = dbutils.secrets.get(scope="kv-scope", key="sp-client-id")
tenant_id     = dbutils.secrets.get(scope="kv-scope", key="sp-tenant-id")
client_secret = dbutils.secrets.get(scope="kv-scope", key="sp-client-secret")

storage_account = "azurelabadls225"

spark.conf.set(f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net", client_secret)

RAW_PATH       = f"abfss://raw@{storage_account}.dfs.core.windows.net"
PROCESSED_PATH = f"abfss://processed@{storage_account}.dfs.core.windows.net"
CURATED_PATH   = f"abfss://curated@{storage_account}.dfs.core.windows.net"

# Enable Adaptive Query Execution — Spark auto-optimizes skew
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")

print("Setup complete")

In [0]:
df_silver = spark.read.format("delta").load(f"{PROCESSED_PATH}/delta/silver_yellow_taxi")
print(f"Silver rows: {df_silver.count():,}")

In [0]:
from pyspark.sql.functions import count, col

# Show trip distribution by borough — this proves Manhattan skew
df_silver.groupBy("pickup_borough") \
    .agg(count("*").alias("trip_count")) \
    .orderBy(col("trip_count").desc()) \
    .show()

In [0]:
from pyspark.sql.functions import (sum, avg, count, round, 
                                    col, lit, rand, floor)

# SKEW HANDLING WITH SALTING
# Manhattan has 80% of trips — one partition gets overloaded
# Fix: add random salt to spread Manhattan across multiple partitions

SALT_BUCKETS = 4  # split each borough into 4 sub-partitions

# Step 1 — Add salt key to Silver data
df_salted = df_silver.withColumn("salt", (floor(rand() * SALT_BUCKETS)).cast("int"))

# Step 2 — First aggregation WITH salt (partial results per salt bucket)
df_partial = (df_salted
    .groupBy("pickup_borough", "pickup_zone_name", "salt")
    .agg(
        count("*").alias("trip_count"),
        sum("fare_amount").alias("total_fare"),
        sum("tip_amount").alias("total_tips"),
        sum("total_amount").alias("total_revenue"),
        avg("trip_distance").alias("avg_distance")
    )
)

# Step 3 — Second aggregation WITHOUT salt (combine partial results)
df_revenue_by_zone = (df_partial
    .groupBy("pickup_borough", "pickup_zone_name")
    .agg(
        sum("trip_count").alias("trip_count"),
        round(sum("total_fare"), 2).alias("total_fare"),
        round(sum("total_tips"), 2).alias("total_tips"),
        round(sum("total_revenue"), 2).alias("total_revenue"),
        round(avg("avg_distance"), 2).alias("avg_distance")
    )
    .orderBy(col("total_revenue").desc())
)

print("Revenue by zone calculated with skew handling")
df_revenue_by_zone.show(10)

In [0]:
from pyspark.sql.functions import sum, avg, count, round, col

df_hourly_demand = (df_silver
    .groupBy("pickup_hour", "pickup_day")
    .agg(
        count("*").alias("trip_count"),
        round(avg("fare_amount"), 2).alias("avg_fare"),
        round(avg("trip_duration_mins"), 2).alias("avg_duration_mins"),
        round(sum("total_amount"), 2).alias("total_revenue")
    )
    .orderBy("pickup_hour", "pickup_day")
)

print("Hourly demand pattern calculated")
df_hourly_demand.show(10)

In [0]:
from pyspark.sql.functions import sum, avg, count, round, col, when

df_payment = (df_silver
    .withColumn("payment_type_name",
        when(col("payment_type") == 1, "Credit Card")
        .when(col("payment_type") == 2, "Cash")
        .when(col("payment_type") == 3, "No Charge")
        .when(col("payment_type") == 4, "Dispute")
        .otherwise("Unknown"))
    .groupBy("payment_type_name")
    .agg(
        count("*").alias("trip_count"),
        round(avg("fare_amount"), 2).alias("avg_fare"),
        round(avg("tip_amount"), 2).alias("avg_tip"),
        round(sum("total_amount"), 2).alias("total_revenue")
    )
    .orderBy(col("trip_count").desc())
)

print("Payment analysis complete")
df_payment.show()

In [0]:
gold_path = f"{CURATED_PATH}/delta/gold_nyc_taxi"

# Union all KPIs into separate Gold tables
df_revenue_by_zone.write.format("delta").mode("overwrite") \
    .save(f"{CURATED_PATH}/delta/gold_revenue_by_zone")

df_hourly_demand.write.format("delta").mode("overwrite") \
    .save(f"{CURATED_PATH}/delta/gold_hourly_demand")

df_payment.write.format("delta").mode("overwrite") \
    .save(f"{CURATED_PATH}/delta/gold_payment_analysis")

print("All Gold tables written!")

In [0]:
# Z-ORDER clusters related data together in files
# Queries filtering by borough + hour will scan far fewer files

spark.sql(f"""
    OPTIMIZE delta.`{CURATED_PATH}/delta/gold_revenue_by_zone`
    ZORDER BY (pickup_borough, pickup_zone_name)
""")

spark.sql(f"""
    OPTIMIZE delta.`{CURATED_PATH}/delta/gold_hourly_demand`
    ZORDER BY (pickup_hour, pickup_day)
""")

print("Z-ORDER optimization complete!")

In [0]:
df_gold = spark.read.format("delta").load(f"{CURATED_PATH}/delta/gold_revenue_by_zone")
print(f"Gold revenue table rows: {df_gold.count()}")

df_gold_hourly = spark.read.format("delta").load(f"{CURATED_PATH}/delta/gold_hourly_demand")
print(f"Gold hourly table rows: {df_gold_hourly.count()}")

df_gold_payment = spark.read.format("delta").load(f"{CURATED_PATH}/delta/gold_payment_analysis")
print(f"Gold payment table rows: {df_gold_payment.count()}")

print("\nFULL PIPELINE COMPLETE!")
print("Bronze → Silver → Gold")